# ResNet18 ex novo v3.1 - Waste Type Identification

Esperimento basato su ResNet18 pretrained ImageNet. Versione "course-only": utilizza esclusivamente tecniche ML riconducibili agli argomenti del corso (Slides + Laboratorio), rimuovendo ottimizzazioni hardware e tecniche non trattate (AMP, cudnn benchmark, non_blocking).

Scelte principali:

- ResNet18 con fine-tuning completo e learning rate differenziati tra backbone e classificatore.
- Split train/validation stratificato 80/20.
- Loss pesata con pesi morbidi calcolati sulle classi del train set, label smoothing 0.1 e WeightedRandomSampler per bilanciare le classi minoritarie.
- Augmentation moderata standard (flip, rotazione, ColorJitter, RandomErasing) e regolarizzazione con dropout, senza affine/shear/zoom aggressivi.
- Learning rate adattivo (ReduceLROnPlateau), early stopping e model selection su balanced accuracy di validation.
- Training in FP32 puro, senza mixed precision.
- Ogni run salva risultati leggeri in `results/<run_name>/`.
- Checkpoint salvato su Google Drive, non su GitHub.

Questo notebook usa solo dal notebook del collega le informazioni sul path del dataset: `MyDrive/waste_type_identification.zip` e/o `MyDrive/waste_type_identification`.

## 1. Mount Drive e clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import datetime
import subprocess
from getpass import getpass

REPO_OWNER = 'fabriziodrr'
REPO_NAME = 'ProjectML'
BASE_BRANCH = 'exp/resnet18-exnovo-v3.1'
RUN_ID = datetime.datetime.now().strftime('%Y%m%d-%H%M')
RUN_BRANCH = f'exp-resnet18-exnovo-v31-{RUN_ID}'
RUN_NAME = f'resnet18_exnovo_v31_{RUN_ID}'
REPO_PATH = f'/content/{REPO_NAME}'
PUBLIC_REPO_URL = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'

def sanitize(text):
    if 'AUTH_REPO_URL' in globals():
        text = text.replace(AUTH_REPO_URL, 'https://***@github.com/' + REPO_OWNER + '/' + REPO_NAME + '.git')
    if 'GITHUB_TOKEN' in globals() and GITHUB_TOKEN:
        text = text.replace(GITHUB_TOKEN, '***')
    return text

def run(cmd, cwd=None, check=True):
    printable = sanitize(' '.join(cmd))
    print('$', printable)
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(sanitize(result.stdout))
    if result.stderr:
        print(sanitize(result.stderr))
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {printable}')
    return result

GITHUB_TOKEN = getpass('GitHub token personale con permessi repo: ')
AUTH_REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'

if not os.path.exists(REPO_PATH):
    run(['git', 'clone', '--branch', BASE_BRANCH, AUTH_REPO_URL, REPO_PATH])
else:
    print('Repo gia presente nel runtime.')

run(['git', 'remote', 'set-url', 'origin', AUTH_REPO_URL], cwd=REPO_PATH)
run(['git', 'pull', '--rebase', 'origin', BASE_BRANCH], cwd=REPO_PATH)
run(['git', 'remote', 'set-url', 'origin', PUBLIC_REPO_URL], cwd=REPO_PATH)
branch_exists = subprocess.run(['git', 'rev-parse', '--verify', RUN_BRANCH], cwd=REPO_PATH, text=True, capture_output=True).returncode == 0
if branch_exists:
    run(['git', 'switch', RUN_BRANCH], cwd=REPO_PATH)
else:
    run(['git', 'switch', '-c', RUN_BRANCH], cwd=REPO_PATH)
os.chdir(REPO_PATH)
print('Working directory:', os.getcwd())
print('Run branch:', RUN_BRANCH)
print('Run name:', RUN_NAME)

## 2. Configurazione esperimento

In [ ]:
import json
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
from torchvision import models, transforms
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

EXP_NAME = 'resnet18_exnovo_v31'
SEED = 42
NUM_CLASSES = 8
BATCH_SIZE = 32
NUM_WORKERS = 0
VAL_SIZE = 0.20
EPOCHS = 35
PATIENCE = 8
BACKBONE_LR = 1e-4
HEAD_LR = 5e-4
DROPOUT_P = 0.25
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1

DRIVE_ZIP_PATH = '/content/drive/MyDrive/waste_type_identification.zip'
DRIVE_DATASET_DIR = '/content/drive/MyDrive/waste_type_identification'
LOCAL_DATASET_DIR = '/content/waste_type_identification'

RESULTS_DIR = Path(REPO_PATH) / 'results' / RUN_NAME
DRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/ProjectML_checkpoints') / RUN_NAME

CLASS_NAMES = ['Battery', 'Clothing', 'Glass', 'Metal', 'Organic', 'Papery', 'Plastic', 'Undifferentiated']
LABEL_MAP = {
    'battery': 0,
    'clothes': 1, 'clothing': 1, 'shoes': 1,
    'brown': 2, 'green': 2, 'transparent': 2, 'glass': 2,
    'metal': 3,
    'organic': 4,
    'cardboard': 5, 'paper': 5, 'papery': 5,
    'plastic': 6,
    'undifferentiated': 7,
}
IMG_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Preparazione dataset

In [ ]:
local_dataset = Path(LOCAL_DATASET_DIR)
drive_dataset = Path(DRIVE_DATASET_DIR)
drive_zip = Path(DRIVE_ZIP_PATH)

if not local_dataset.exists():
    if drive_dataset.exists():
        print('Copio dataset gia estratto da Drive a disco locale Colab...')
        shutil.copytree(drive_dataset, local_dataset)
    elif drive_zip.exists():
        print('Estraggo zip dataset da Drive nella cartella locale attesa...')
        local_dataset.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(drive_zip, 'r') as zf:
            zf.extractall(local_dataset)
    else:
        raise FileNotFoundError(f'Non trovo ne {drive_dataset} ne {drive_zip}')

if not local_dataset.exists():
    candidates = [p for p in Path('/content').iterdir() if p.is_dir() and 'waste' in p.name.lower()]
    print('Candidate dataset folders:', candidates)
    raise FileNotFoundError(f'Dataset locale atteso non trovato: {local_dataset}')

print('Dataset locale:', local_dataset)
print('Prime cartelle:', sorted([p.name for p in local_dataset.iterdir() if p.is_dir()])[:30])

In [ ]:
samples = []
scanned_folders = set()
skipped_folders = set()
for dirpath, _, filenames in os.walk(local_dataset, followlinks=False):
    folder = Path(dirpath).name.lower()
    if not filenames:
        continue
    scanned_folders.add(folder)
    if folder not in LABEL_MAP:
        skipped_folders.add(folder)
        continue
    label = LABEL_MAP[folder]
    for fname in filenames:
        if fname.lower().endswith(IMG_EXTENSIONS):
            path = Path(dirpath) / fname
            samples.append({
                'path': str(path),
                'relative_path': str(path.relative_to(local_dataset)),
                'folder': folder,
                'label': label,
                'class_name': CLASS_NAMES[label],
            })

if skipped_folders:
    print(f'WARNING: {len(skipped_folders)} cartelle ignorate (non in LABEL_MAP): {sorted(skipped_folders)}')

df = pd.DataFrame(samples).sort_values('relative_path').reset_index(drop=True)
if df.empty:
    raise RuntimeError('Nessuna immagine trovata. Controlla LABEL_MAP e struttura dataset.')

print('Totale immagini:', len(df))
print(df['class_name'].value_counts().reindex(CLASS_NAMES, fill_value=0))

splitter = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
train_idx, val_idx = next(splitter.split(df['relative_path'], df['label']))
df['split'] = 'train'
df.loc[val_idx, 'split'] = 'val'

split_path = RESULTS_DIR / 'split.csv'
df[['relative_path', 'folder', 'label', 'class_name', 'split']].to_csv(split_path, index=False)
print('Split salvato in:', split_path)
print(pd.crosstab(df['class_name'], df['split']).reindex(CLASS_NAMES, fill_value=0))

## 4. Dataset, transform e DataLoader

In [ ]:
class WasteDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row['label'])

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.80, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.15, hue=0.03),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.08), ratio=(0.3, 3.3)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df = df[df['split'] == 'val'].reset_index(drop=True)

sample_counts = train_df['label'].value_counts().sort_index().to_numpy()
sample_inv_weights = 1.0 / np.maximum(sample_counts, 1.0)
train_sample_weights = sample_inv_weights[train_df['label'].values]
train_sampler = WeightedRandomSampler(train_sample_weights, num_samples=len(train_sample_weights), replacement=True)

train_loader = DataLoader(WasteDataset(train_df, train_transform), batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(WasteDataset(val_df, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

train_counts = train_df['label'].value_counts().reindex(range(NUM_CLASSES), fill_value=0).sort_index().to_numpy(dtype=np.float32)
class_weights = np.sqrt(train_counts.max() / np.maximum(train_counts, 1.0))
class_weights = class_weights / class_weights.mean()
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

weights_df = pd.DataFrame({'label': range(NUM_CLASSES), 'class_name': CLASS_NAMES, 'train_count': train_counts.astype(int), 'loss_weight': class_weights})
weights_df.to_csv(RESULTS_DIR / 'class_weights.csv', index=False)
print(weights_df)

## 5. Modello ResNet18

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=DROPOUT_P),
    nn.Linear(num_features, NUM_CLASSES),
)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=LABEL_SMOOTHING)
backbone_params = [p for name, p in model.named_parameters() if not name.startswith('fc.')]
head_params = list(model.fc.parameters())
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': BACKBONE_LR},
    {'params': head_params, 'lr': HEAD_LR},
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)

config = {
    'experiment': EXP_NAME,
    'run_id': RUN_ID,
    'run_name': RUN_NAME,
    'base_branch': BASE_BRANCH,
    'run_branch': RUN_BRANCH,
    'model': 'torchvision.models.resnet18 pretrained ImageNet',
    'strategy': 'full_finetuning_with_discriminative_learning_rates_FP32_no_AMP',
    'seed': SEED,
    'val_size': VAL_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'patience': PATIENCE,
    'backbone_lr': BACKBONE_LR,
    'head_lr': HEAD_LR,
    'dropout_p': DROPOUT_P,
    'weight_decay': WEIGHT_DECAY,
    'loss': 'CrossEntropyLoss with sqrt inverse frequency class weights and label smoothing 0.1',
    'augmentation': 'Resize, moderate RandomResizedCrop, horizontal flip, small rotation, light ColorJitter, RandomErasing; no affine/shear/strong zoom; no per-class augmentation; WeightedRandomSampler for class balancing',
      'course_arguments': 'transfer learning, learning-rate scheduling (ReduceLROnPlateau), early stopping, L2 regularization, dropout, data augmentation, label smoothing, class balancing (WeightedRandomSampler), balanced accuracy',
    'dataset_drive_zip': DRIVE_ZIP_PATH,
    'dataset_drive_dir': DRIVE_DATASET_DIR,
    'results_dir': str(RESULTS_DIR),
    'checkpoint_drive_dir': str(DRIVE_CHECKPOINT_DIR),
}
with open(RESULTS_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))

## 6. Training e validazione

In [ ]:
def run_epoch(model, loader, train):
    model.train(train)
    total_loss = 0.0
    all_preds = []
    all_targets = []

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, targets in loader:
            images = images.to(DEVICE)
            targets = targets.to(DEVICE)

            if train:
                optimizer.zero_grad()

                logits = model(images)
                loss = criterion(logits, targets)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            all_preds.append(logits.argmax(dim=1).detach().cpu())
            all_targets.append(targets.detach().cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_targets).numpy()
    loss = total_loss / len(loader.dataset)
    acc = float((y_pred == y_true).mean())
    bal_acc = float(balanced_accuracy_score(y_true, y_pred))
    return loss, acc, bal_acc, y_true, y_pred

best_bal_acc = -1.0
best_epoch = -1
epochs_without_improvement = 0
history = []
best_checkpoint_path = DRIVE_CHECKPOINT_DIR / 'resnet18_exnovo_v31_best.pth'

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, train_bal_acc, _, _ = run_epoch(model, train_loader, train=True)
    val_loss, val_acc, val_bal_acc, y_true, y_pred = run_epoch(model, val_loader, train=False)

    # Log LR actually used for this epoch (from optimizer, before scheduler step)
    row = {
        'epoch': epoch,
        'backbone_lr': optimizer.param_groups[0]['lr'],
        'head_lr': optimizer.param_groups[1]['lr'],
        'train_loss': train_loss,
        'train_acc': train_acc,
        'train_bal_acc': train_bal_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'val_bal_acc': val_bal_acc,
    }
    history.append(row)

    print(f'Epoch {epoch:02d}: lr=[{row["backbone_lr"]:.2e},{row["head_lr"]:.2e}] train_loss={train_loss:.4f} train_ba={train_bal_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_ba={val_bal_acc:.4f}')

    if val_bal_acc > best_bal_acc:
        best_bal_acc = val_bal_acc
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_names': CLASS_NAMES,
            'config': config,
            'best_epoch': best_epoch,
            'best_val_bal_acc': best_bal_acc,
        }, best_checkpoint_path)
        print('  Saved best checkpoint to Drive:', best_checkpoint_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print('Early stopping: balanced accuracy non migliora da', PATIENCE, 'epoche.')
            break

    scheduler.step(val_bal_acc)

# Write history once at end of training
pd.DataFrame(history).to_csv(RESULTS_DIR / 'history.csv', index=False)

summary = {'run_name': RUN_NAME, 'run_branch': RUN_BRANCH, 'results_dir': str(RESULTS_DIR), 'best_epoch': best_epoch, 'best_val_bal_acc': best_bal_acc, 'checkpoint_path_drive': str(best_checkpoint_path)}
with open(RESULTS_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Summary:', summary)

## 7. Metriche finali e grafici

In [ ]:
checkpoint = torch.load(best_checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

val_loss, val_acc, val_bal_acc, y_true, y_pred = run_epoch(model, val_loader, train=False)
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

with open(RESULTS_DIR / 'classification_report.txt', 'w') as f:
    f.write(report)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(RESULTS_DIR / 'confusion_matrix.csv')

print(report)
print('Final val accuracy:', val_acc)
print('Final val balanced accuracy:', val_bal_acc)

history_df = pd.read_csv(RESULTS_DIR / 'history.csv')
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_df['epoch'], history_df['train_loss'], label='train')
plt.plot(history_df['epoch'], history_df['val_loss'], label='val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('ResNet18 loss')
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_df['epoch'], history_df['train_bal_acc'], label='train BA')
plt.plot(history_df['epoch'], history_df['val_bal_acc'], label='val BA')
plt.xlabel('Epoch')
plt.ylabel('Balanced accuracy')
plt.title('ResNet18 balanced accuracy')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=160)
plt.show()

plt.figure(figsize=(8, 7))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('ResNet18 confusion matrix')
plt.colorbar()
ticks = np.arange(NUM_CLASSES)
plt.xticks(ticks, CLASS_NAMES, rotation=45, ha='right')
plt.yticks(ticks, CLASS_NAMES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=160)
plt.show()

# Normalized confusion matrix (row-wise, percentages)
cm_norm = cm.astype('float64') / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)
plt.figure(figsize=(8, 7))
plt.imshow(cm_norm, interpolation='nearest', cmap='Greens', vmin=0, vmax=1)
plt.title('ResNet18 confusion matrix (normalized by row)')
plt.colorbar()
plt.xticks(ticks, CLASS_NAMES, rotation=45, ha='right')
plt.yticks(ticks, CLASS_NAMES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center', fontsize=8,
                 color='white' if cm_norm[i, j] > 0.5 else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix_norm.png', dpi=160)
plt.show()

## 8. Note esperimento

In [ ]:
notes = f'''# {RUN_NAME}

## Ipotesi
ResNet18 pretrained puo dare una baseline forte con costo computazionale contenuto. Questa variante (v3.1) utilizza ESCLUSIVAMENTE tecniche ML riconducibili agli argomenti del corso: transfer learning, learning rate differenziati, regolarizzazione L2, dropout, data augmentation moderata, learning rate decrescente ed early stopping.

## Modifiche v3.1 rispetto a v3 (rimozione tecniche non corso)
- RIMOSSO AMP (Automatic Mixed Precision): training in FP32 puro come da corso (S:07-08 Back Propagation, L:03-04 PyTorch base).
- RIMOSSO torch.backends.cudnn.benchmark: nessun riferimento nel corso.
- RIMOSSO non_blocking=True nei trasferimenti GPU: tecnica di ottimizzazione CUDA non trattata.
- RIMOSSO set_to_none=True in zero_grad(): micro-ottimizzazione non trattata.
- CORRETTO ordinamento RandomErasing: ora prima di Normalize (posizione standard).
- Training loop semplificato: loss.backward() + optimizer.step() diretti, senza scaler.

## Tecniche del corso mantenute in v3.1
- Transfer learning / Fine-tuning: S:12 (teoria) + L:08 (AlexNet fine-tuning)
- ResNet18 / Skip connections: S:14 (Residual Learning)
- StratifiedShuffleSplit: L:01 (KNN validation)
- Data augmentation (flip, rotazione, ColorJitter, RandomErasing): S:05 (teoria) + L:04 (pratica)
- AdamW optimizer: da S:09 (Adam) + L:04 (Adam pratica)
- CrossEntropyLoss con class weights: S:09 (CCE) + L:04 (pratica)
- Label smoothing: S:09 (regolarizzazione loss)
- WeightedRandomSampler: S:04-05 (class imbalance)
- ReduceLROnPlateau: S:09 (learning rate decay)
- Early stopping: S:09 (teoria) + L:04 (pratica)
- Dropout: S:13 (teoria) + L:07A (AlexNet classifier)
- L2 regularization (weight_decay): S:09 (teoria)
- Balanced accuracy: da S:05 (recall per classe)
- Discriminative learning rates: da S:12 (transfer learning)
- ImageNet normalization: L:07A (AlexNet preprocessing)

## Differenze rispetto al notebook del collega
- Niente augmentation per-classe.
- WeightedRandomSampler per bilanciare le classi nel training.
- Label smoothing 0.1 per ridurre overconfidence.
- Niente training in due fasi frozen/unfrozen.
- Niente affine/shear/zoom aggressivi.
- Training in FP32 puro (no AMP).

## Risultato
Best epoch: {best_epoch}
Best validation balanced accuracy: {best_bal_acc:.4f}
Branch run: {RUN_BRANCH}
Cartella risultati: {RESULTS_DIR}
Checkpoint Drive: {best_checkpoint_path}

## File prodotti in repo
- `split.csv`
- `config.json`
- `class_weights.csv`
- `history.csv`
- `summary.json`
- `classification_report.txt`
- `confusion_matrix.csv`
- `confusion_matrix.png`
- `confusion_matrix_norm.png`
- `training_curves.png`
'''
with open(RESULTS_DIR / 'notes.md', 'w') as f:
    f.write(notes)
print(notes)


## 9. Commit e push risultati su GitHub

In [ ]:
os.chdir(REPO_PATH)
run(['git', 'config', 'user.name', 'fabriziodrr'], cwd=REPO_PATH)
run(['git', 'config', 'user.email', 'fabriziodrr@users.noreply.github.com'], cwd=REPO_PATH)

run(['git', 'add', str(RESULTS_DIR.relative_to(REPO_PATH)), 'src/resnet18_exnovo/resnet18_exnovo_colab.ipynb'], cwd=REPO_PATH)
status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_PATH).decode().strip()

if not status:
    print('Nessuna modifica da committare.')
else:
    message = f'Add ResNet18 exnovo v3 results {RUN_ID}'
    run(['git', 'commit', '-m', message], cwd=REPO_PATH)
    auth_url = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
    try:
        run(['git', 'remote', 'set-url', 'origin', auth_url], cwd=REPO_PATH)
        run(['git', 'push', '-u', 'origin', RUN_BRANCH], cwd=REPO_PATH)
    finally:
        run(['git', 'remote', 'set-url', 'origin', PUBLIC_REPO_URL], cwd=REPO_PATH, check=False)

print('Risultati sincronizzati su GitHub.')